# Elmatrix: render from your iPhone
Run these cells in order in an interactive Colab CPU runtime. Upload only your script-only job ZIP. The notebook creates an original title card and generic synthetic English narration, then renders a video. Your script goes to Google Colab. No YouTube credentials, Drive mount, paid API or automatic publication. Preview the output. Free runtime access is not guaranteed. This is a basic narrated-card draft, not generated footage.


In [ ]:
import subprocess, sys
# Runs in the Colab VM, NOT in iSH. CPU runtime is sufficient.
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg', 'espeak-ng', 'fonts-dejavu-core'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'Pillow==12.3.0'], check=True)


In [ ]:
# Keep renderer source as data; execute it only in a fresh Python process.
_ELMATRIX_RENDERER_SOURCE = '"""Data-only handoff for iSH; rendering happens in an interactive cloud notebook."""\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport tempfile\nimport textwrap\nimport zipfile\n\nclass CloudError(ValueError):\n    pass\n\n\nMAX_RESULT = 128 * 1024 * 1024\n\n\ndef is_ish():\n    return hasattr(os, \'uname\') and \'ish\' in os.uname().release.lower()\n\n\ndef canonical(data):\n    return json.dumps(data, sort_keys=True, ensure_ascii=False).encode(\'utf-8\')\n\n\ndef validated(data):\n    if not isinstance(data, dict) or data.get(\'schema\') != 1:\n        raise CloudError(\'Unsupported project schema\')\n    result = {\'schema\': 1}\n    for key, limit in [(\'title\', 100), (\'description\', 4500), (\'script\', 2000)]:\n        value = data.get(key)\n        if (not isinstance(value, str) or not value.strip() or len(value) > limit\n                or any(ord(c) < 32 and c not in \'\\n\\t\' for c in value)\n                or \'<\' in value or \'>\' in value):\n            raise CloudError(\'Invalid \' + key)\n        result[key] = value\n    # This first synthetic voice/card implementation is English-only.\n    if not result[\'script\'].isascii() or not result[\'title\'].isascii():\n        raise CloudError(\'Cloud draft currently requires an English ASCII title and script\')\n    return result\n\n\ndef read_project(root):\n    path = Path(root) / \'project.json\'\n    if path.stat().st_size > 65536:\n        raise CloudError(\'Project too large\')\n    return validated(json.loads(path.read_text(encoding=\'utf-8\')))\n\n\ndef fingerprint(data):\n    return hashlib.sha256(canonical(validated(data))).hexdigest()\n\n\ndef exclusive_json(path, data):\n    fd = os.open(path, os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o600)\n    with os.fdopen(fd, \'wb\') as out:\n        out.write(canonical(data)); out.flush(); os.fsync(out.fileno())\n\n\ndef prepare(root, destination):\n    data = read_project(root)\n    # Export only explicitly selected content. Never archive the project directory.\n    with zipfile.ZipFile(destination, \'x\', compression=zipfile.ZIP_STORED) as archive:\n        archive.writestr(\'project.json\', canonical(data))\n    print(\'Prepared script-only bundle:\', destination)\n    print(\'No credentials, vault files or local media included. Upload it to the notebook.\')\n\n\ndef checked_members(archive, names, limit):\n    entries = archive.infolist()\n    if len(entries) != len(names) or {i.filename for i in entries} != set(names):\n        raise CloudError(\'Unexpected, duplicate or missing archive entries\')\n    if sum(i.file_size for i in entries) > limit:\n        raise CloudError(\'Archive exceeds size limit\')\n    for i in entries:\n        mode = i.external_attr >> 16\n        if (i.flag_bits & 1 or i.is_dir() or\n                (mode & 0o170000) not in (0, 0o100000)):\n            raise CloudError(\'Archive contains a nonregular or encrypted member\')\n        if i.filename.endswith(\'.json\') and i.file_size > 65536:\n            raise CloudError(\'Metadata too large\')\n\n\ndef file_hash(path):\n    h = hashlib.sha256()\n    with open(path, \'rb\') as source:\n        for block in iter(lambda: source.read(1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef run_cloud(bundle, output):\n    """Run on full Linux only, after notebook user explicitly starts the cell."""\n    if is_ish():\n        raise CloudError(\'Cloud renderer must run in the notebook, not iSH\')\n    for tool in (\'ffmpeg\', \'ffprobe\', \'espeak-ng\'):\n        if not shutil.which(tool):\n            raise CloudError(\'Cloud runtime missing \' + tool + \'; run the setup cell\')\n    from PIL import Image, ImageDraw, ImageFont\n    with zipfile.ZipFile(bundle) as archive:\n        checked_members(archive, [\'project.json\'], 65536)\n        data = validated(json.loads(archive.read(\'project.json\')))\n    with tempfile.TemporaryDirectory(prefix=\'elmatrix-cloud-\') as directory:\n        root = Path(directory)\n        image = Image.new(\'RGB\', (720, 1280), \'#101b2d\')\n        draw = ImageDraw.Draw(image)\n        font_path = \'/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf\'\n        heading = ImageFont.truetype(font_path, 46)\n        small = ImageFont.truetype(font_path, 26)\n        draw.rounded_rectangle((46, 70, 674, 1210), radius=30, outline=\'#45d6bb\', width=4)\n        draw.text((82, 140), \'SERVICE CHECKLIST\', font=small, fill=\'#45d6bb\')\n        y = 270\n        for line in textwrap.wrap(data[\'title\'], width=20):\n            draw.text((82, y), line, font=heading, fill=\'white\'); y += 66\n        draw.text((82, 1080), \'Listen to the checklist\', font=small, fill=\'#b8cadf\')\n        cover = root / \'cover.png\'; image.save(cover)\n        audio = root / \'narration.wav\'\n        subprocess.run([\'espeak-ng\', \'-v\', \'en\', \'-s\', \'155\', \'-w\', str(audio), \'--stdin\'],\n                       input=data[\'script\'].encode(), check=True, timeout=120,\n                       stdout=subprocess.PIPE, stderr=subprocess.PIPE)\n        probe = subprocess.run([\'ffprobe\', \'-v\', \'error\', \'-show_entries\', \'format=duration\',\n            \'-of\', \'json\', str(audio)], check=True, capture_output=True, text=True, timeout=30)\n        duration = float(json.loads(probe.stdout)[\'format\'][\'duration\'])\n        if not 1 <= duration <= 180:\n            raise CloudError(\'Narration exceeds pilot duration limit; shorten the script\')\n        video = root / \'video.mp4\'\n        subprocess.run([\'ffmpeg\', \'-nostdin\', \'-v\', \'error\', \'-n\', \'-loop\', \'1\',\n            \'-i\', str(cover), \'-i\', str(audio), \'-map\', \'0:v:0\', \'-map\', \'1:a:0\',\n            \'-c:v\', \'libx264\', \'-preset\', \'veryfast\', \'-tune\', \'stillimage\',\n            \'-pix_fmt\', \'yuv420p\', \'-r\', \'24\', \'-c:a\', \'aac\', \'-b:a\', \'128k\',\n            \'-t\', str(duration), \'-shortest\', \'-movflags\', \'+faststart\', str(video)],\n            check=True, timeout=1800, stdout=subprocess.PIPE, stderr=subprocess.PIPE)\n        probe = subprocess.run([\'ffprobe\', \'-v\', \'error\', \'-show_entries\',\n            \'format=duration:stream=codec_type,width,height\', \'-of\', \'json\', str(video)],\n            check=True, capture_output=True, text=True, timeout=30)\n        info = json.loads(probe.stdout)\n        if {s[\'codec_type\'] for s in info[\'streams\']} != {\'video\', \'audio\'}:\n            raise CloudError(\'Output is missing video/audio\')\n        if video.stat().st_size > MAX_RESULT - 65536:\n            raise CloudError(\'Rendered file exceeds return-bundle limit\')\n        manifest = {\'schema\': 1, \'project_sha256\': fingerprint(data),\n            \'sha256\': file_hash(video), \'duration\': duration, \'format\': \'720x1280 H.264/AAC\',\n            \'voice\': \'espeak-ng en; synthetic, not an imitation of a person\',\n            \'renderer\': \'cloud-draft-v1\'}\n        with zipfile.ZipFile(output, \'x\', compression=zipfile.ZIP_STORED) as archive:\n            archive.write(video, \'video.mp4\')\n            archive.writestr(\'render.json\', canonical(manifest))\n    return Path(output)\n\n\ndef import_result(root, bundle):\n    root = Path(root).resolve(strict=True)\n    data = read_project(root)\n    for name in (\'video.mp4\', \'render.json\', \'upload-attempt.json\'):\n        if os.path.lexists(root / name):\n            raise CloudError(\'Project already has render/upload state; use a new project for revisions\')\n    with tempfile.TemporaryDirectory(prefix=\'.cloud-import-\', dir=root) as directory:\n        stage = Path(directory) / \'video.mp4\'\n        with zipfile.ZipFile(bundle) as archive:\n            checked_members(archive, [\'video.mp4\', \'render.json\'], MAX_RESULT)\n            manifest = json.loads(archive.read(\'render.json\'))\n            if (not isinstance(manifest, dict) or manifest.get(\'schema\') != 1 or\n                    manifest.get(\'project_sha256\') != fingerprint(data)):\n                raise CloudError(\'Result does not match this project; nothing imported\')\n            h = hashlib.sha256(); total = 0\n            with archive.open(\'video.mp4\') as source, stage.open(\'xb\') as out:\n                while True:\n                    block = source.read(1024 * 1024)\n                    if not block: break\n                    total += len(block)\n                    if total > MAX_RESULT: raise CloudError(\'Video too large\')\n                    h.update(block); out.write(block)\n                out.flush(); os.fsync(out.fileno())\n            if h.hexdigest() != manifest.get(\'sha256\'):\n                raise CloudError(\'Result hash mismatch; nothing imported\')\n        with stage.open(\'rb\') as stream:\n            header = stream.read(12)\n        if len(header) < 12 or header[4:8] != b\'ftyp\':\n            raise CloudError(\'Not an MP4 container\')\n        os.chmod(stage, 0o600)\n        os.link(stage, root / \'video.mp4\')\n        # A metadata-write failure leaves the video, blocks retry, and never enables upload.\n        exclusive_json(root / \'render.json\', manifest)\n    print(\'Imported verified bundle into\', root)\n    print(\'Hash checks are not proof of trusted content. Watch the video before any upload.\')\n\n\ndef doctor():\n    if is_ish():\n        return {\'platform\': \'ish\', \'local_render\': \'disabled\',\n                \'reason\': \'FFmpeg SIGSYS reported on iSH; no native binaries probed\',\n                \'next\': \'prepare-cloud, render in browser notebook, import-render\',\n                \'cloud_execution\': \'requires your interactive Google Colab session\',\n                \'automatic_upload\': False}\n    return {\'platform\': \'other\', \'local_render\': \'not probed\',\n            \'ffmpeg_on_path\': bool(shutil.which(\'ffmpeg\')),\n            \'ffprobe_on_path\': bool(shutil.which(\'ffprobe\')),\n            \'note\': \'Presence does not prove execution compatibility\'}\n'


In [ ]:
from pathlib import Path
import tempfile, zipfile, subprocess, sys
from google.colab import files
from IPython.display import Video, display
# Select only the script-only ZIP produced by prepare-cloud.
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Choose exactly one job ZIP')
content = next(iter(uploaded.values()))
if len(content) > 1024 * 1024:
    raise ValueError('Job bundle too large')
job_dir = Path(tempfile.mkdtemp(prefix='elmatrix-notebook-'))
bundle = job_dir / 'job.zip'
bundle.write_bytes(content)
renderer = job_dir / 'cloud_video.py'
renderer.write_text(_ELMATRIX_RENDERER_SOURCE, encoding='utf-8')
result = job_dir / 'render-result.zip'
# Isolated child avoids stale Pillow modules already loaded in the notebook kernel.
subprocess.run([sys.executable, '-I', '-c',
    "import runpy,sys; runpy.run_path(sys.argv[1])['run_cloud'](sys.argv[2],sys.argv[3])",
    str(renderer), str(bundle), str(result)], check=True, timeout=2100)
# Extract fixed video name for preview, never arbitrary archive paths.
with zipfile.ZipFile(result) as archive:
    preview = job_dir / 'preview.mp4'
    preview.write_bytes(archive.read('video.mp4'))
display(Video(str(preview), embed=True))
files.download(str(result))
print('Save render-result.zip to xxix in Files, then import-render in iSH.')
print('No YouTube upload or channel authorization happened.')
